# Customer360 — Universal RFM Pipeline
### Customer-Level Segmentation — Works With Any Transaction Dataset

**Pipeline:** Upload → Schema Detection → EDA → Clean → **RFM Aggregation** → Scale → PCA → Optimal K → **K-Means + GMM + Hierarchical** → Best Algorithm → **RFM Segment Labels** → SHAP → Report

> Upload any CSV with sales/transaction data. The pipeline aggregates to **customer level**, computes RFM features, runs three clustering algorithms, picks the best by Silhouette Score, and assigns industry-standard behavioural segment names.

## Cell 1: Install Libraries & Import Everything

In [ ]:
!pip install -q shap fpdf2 plotly kaleido squarify

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                              davies_bouldin_score, calinski_harabasz_score,
                              adjusted_rand_score)
from sklearn.ensemble import RandomForestClassifier
from scipy import stats
from collections import Counter
import shap
import plotly.express as px
import squarify
from fpdf import FPDF
import warnings, os
from datetime import datetime, timedelta
from google.colab import files

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 150, 'font.family': 'sans-serif',
                     'axes.titleweight': 'bold', 'axes.titlesize': 13})

OUTPUT_DIR = '/content/customer360_results'
os.makedirs(f'{OUTPUT_DIR}/charts', exist_ok=True)
print('✅ All libraries loaded successfully!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.2 MB/s eta 0:00:00
✅ All libraries loaded successfully!


## Cell 2: Upload Your Dataset

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Try multiple encodings
raw_df = None
for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        raw_df = pd.read_csv(filename, encoding=enc)
        print(f'✅ Loaded with encoding: {enc}')
        break
    except Exception:
        continue
if raw_df is None:
    raise ValueError('Could not read CSV. Try saving it as UTF-8.')

print(f'✅ Loaded: {len(raw_df):,} rows from "{filename}"')
print(f'Columns ({len(raw_df.columns)}): {list(raw_df.columns)}')
raw_df.head(3)

Saving threetwentyone.csv to threetwentyone.csv
✅ Loaded with encoding: utf-8
✅ Loaded: 1,520 rows from "threetwentyone.csv"
Columns (17): ['Order ID (Hashed)', 'Customer ID (Spoofed)', 'Purchase Date', 'Order Status', 'Currency', 'Payment Method', 'Shipping Method', 'Discount Amount', 'Promo Code', 'Customer Country', 'Product Name', 'Categories', 'Color', 'Size', 'Quantity', 'Unit Price', 'Total Line Amount']


,Order ID (Hashed),Customer ID (Spoofed),Purchase Date,Order Status,Currency,Payment Method,Shipping Method,Discount Amount,Promo Code,Customer Country,Product Name,Categories,Color,Size,Quantity,Unit Price,Total Line Amount
0,91c68d9c1b4c,78f7ac159098,2026-02-15T20:50:41.779Z,delivered,GHS,paystack,13,0.0,NaN,GH,Logo Bag,Bags,TAUPE,ONE-SIZE,1,1700.0,1700.0
1,91c68d9c1b4c,78f7ac159098,2026-02-15T20:50:41.779Z,delivered,GHS,paystack,13,0.0,NaN,GH,Desire Vest,T-Shirts; Tanks,BROWN,M,1,935.0,935.0
2,72db6e5f4a03,51b786753fcf,2026-02-09T23:25:43.254Z,confirmed,GBP,paystack,13,0.0,NaN,GH,Arch Tote Obolo,Bags,BLACK,ONE-SIZE,1,2550.0,2550.0


## Cell 3: Auto-Detect Schema

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: detect_schema
# ══════════════════════════════════════════════
def detect_schema(df):
    """Auto-detect column roles from any CSV. Returns dict of column mappings."""
    cols_lower = [c.lower().strip() for c in df.columns]
    orig = list(df.columns)

    def find(keywords, dtype='any'):
        for kw in keywords:
            for i, c in enumerate(cols_lower):
                if kw in c:
                    col = orig[i]
                    if dtype == 'numeric' and not pd.api.types.is_numeric_dtype(df[col]):
                        continue
                    if dtype == 'text' and df[col].dtype != object:
                        continue
                    if dtype == 'date':
                        try:
                            pd.to_datetime(df[col].dropna().iloc[:5])
                        except Exception:
                            continue
                    return col
        return None

    revenue  = (find(['total line amount','total_line','line_amount','total_amount',
                      'order_total','sale_amount','transaction_amount','net_amount'], 'numeric') or
                find(['total','revenue','sales','amount','spend','value','gmv'], 'numeric'))
    price    = (find(['unit price','unit_price','item_price','selling_price',
                      'product_price','sale_price','retail_price'], 'numeric') or
                find(['price','cost','rate','fee'], 'numeric'))
    qty      = (find(['quantity','qty','units','items_sold','units_sold'], 'numeric') or
                find(['count','number','volume'], 'numeric'))
    discount = (find(['discount amount','discount_amount','disc_amount'], 'numeric') or
                find(['discount','markdown','saving'], 'numeric'))
    promo    = (find(['promo code','promo_code','coupon_code','voucher_code'], 'text') or
                find(['promo','coupon','voucher'], 'text'))
    category = (find(['categor','product_type','item_type','department'], 'text') or
                find(['type','class','group'], 'text'))
    product  = (find(['product name','product_name','item_name','sku_name'], 'text') or
                find(['product','item','name','sku'], 'text'))
    payment  = (find(['payment method','payment_method','pay_method'], 'text') or
                find(['payment','pay','tender'], 'text'))
    shipping = (find(['shipping method','ship_method','delivery_method'], 'text') or
                find(['shipping','delivery','fulfillment'], 'text'))
    status   = (find(['order status','order_status','transaction_status'], 'text') or
                find(['status','state'], 'text'))
    date     = (find(['purchase date','purchase_date','order_date','transaction_date',
                      'sale_date','invoice_date'], 'date') or
                find(['date','time','created','timestamp'], 'date'))
    order    = (find(['order id','order_id','transaction_id','invoice_id'], 'text') or
                find(['order','transaction','invoice'], 'text'))

    # Customer ID: explicitly exclude country/address columns
    customer = None
    customer_priority = ['customer id','customer_id','client_id','user_id','member_id','buyer_id']
    customer_fallback = ['customer','client','user','member','buyer']
    for kw_list in [customer_priority, customer_fallback]:
        for kw in kw_list:
            for i, c in enumerate(cols_lower):
                # Exclude columns that are clearly not IDs
                if any(skip in c for skip in ['country','address','city','region','state','zip']):
                    continue
                if kw in c:
                    customer = orig[i]
                    break
            if customer:
                break
        if customer:
            break

    # If revenue missing but qty * price available, compute it
    computed_revenue = False
    if not revenue and qty and price:
        df = df.copy()
        df['_computed_revenue'] = (pd.to_numeric(df[qty], errors='coerce') *
                                   pd.to_numeric(df[price], errors='coerce'))
        revenue = '_computed_revenue'
        computed_revenue = True

    # Auto-detect currency
    currency = 'GHS'
    all_text = ' '.join(orig).lower()
    # Also check currency column values if present
    for col in orig:
        if 'currency' in col.lower() or 'curr' in col.lower():
            top_val = str(df[col].dropna().mode().iloc[0]).upper() if len(df[col].dropna()) > 0 else ''
            all_text += ' ' + top_val.lower()
            break
    currency_map = [
        ('ghs','GHS'),('cedi','GHS'),('ngn','NGN'),('naira','NGN'),
        ('kes','KES'),('shilling','KES'),('zar','ZAR'),('rand','ZAR'),
        ('gbp','GBP'),('pound','GBP'),('eur','EUR'),('euro','EUR'),
        ('usd','USD'),('dollar','USD'),
    ]
    for kw, cur in currency_map:
        if kw in all_text:
            currency = cur
            break

    # Auto-detect business type
    biz_type = 'Retail'
    if category:
        cats = ' '.join(df[category].dropna().astype(str).str.lower().unique()[:50])
        biz_map = [
            (['shirt','dress','trouser','jean','jacket','shoe','bag','fashion','cloth','apparel','wear'], 'Fashion Retail'),
            (['phone','laptop','tablet','electronic','gadget','tech','computer','camera'], 'Electronics Retail'),
            (['food','grocery','beverage','snack','drink','fruit','vegetable','meat','dairy'], 'Food & Grocery'),
            (['beauty','cosmetic','skincare','makeup','hair','fragrance','perfume'], 'Beauty & Personal Care'),
            (['furniture','home','kitchen','decor','bedding','sofa','mattress'], 'Home & Furniture'),
            (['book','stationery','school','education','pen','pencil'], 'Education & Stationery'),
            (['sport','fitness','gym','exercise','outdoor','football'], 'Sports & Fitness'),
        ]
        for keywords, btype in biz_map:
            if any(k in cats for k in keywords):
                biz_type = btype
                break

    return {
        'revenue': revenue, 'price': price, 'qty': qty,
        'discount': discount, 'promo': promo, 'category': category,
        'product': product, 'payment': payment, 'shipping': shipping,
        'status': status, 'date': date, 'order': order, 'customer': customer,
        'currency': currency, 'business': biz_type,
        'computed_revenue': computed_revenue,
    }


SCHEMA = detect_schema(raw_df)
CUR = SCHEMA['currency']
BIZ = SCHEMA['business']

print('=' * 60)
print('  AUTO-DETECTED SCHEMA')
print('=' * 60)
print(f'  Business Type : {BIZ}')
print(f'  Currency      : {CUR}')
print()
for k in ['customer','date','order','revenue','price','qty','discount','promo','category','product','payment','shipping']:
    val = SCHEMA[k]
    icon = '✅' if val else '⚠️  NOT FOUND'
    print(f'  {k:<12}: {val or icon}')
if SCHEMA['computed_revenue']:
    print('  ℹ️  Revenue was missing — computed as qty × unit_price')

# CRITICAL: Customer ID is required for RFM
if not SCHEMA['customer']:
    print()
    print('⚠️  WARNING: No customer ID column detected!')
    print('   Synthetic IDs will be generated from row index.')
    print('   For better results, set: SCHEMA["customer"] = "Your Column Name"')
    raw_df['_synthetic_customer_id'] = 'CUST_' + raw_df.index.astype(str).str.zfill(6)
    SCHEMA['customer'] = '_synthetic_customer_id'

if not SCHEMA['date']:
    raise ValueError('⛔ No date column found. RFM requires a transaction date. Set SCHEMA["date"] = "Your Column Name"')
if not SCHEMA['revenue']:
    raise ValueError('⛔ No revenue column found. Set SCHEMA["revenue"] = "Your Column Name"')

print()
print('✅ Schema detection complete.')
print()
print('# ── MANUAL OVERRIDE (uncomment & edit if anything is wrong) ──')
print('# SCHEMA["customer"] = "Customer ID (Spoofed)"')
print('# SCHEMA["date"]     = "Purchase Date"')
print('# SCHEMA["revenue"]  = "Total Line Amount"')
print('# SCHEMA["order"]    = "Order ID (Hashed)"')
print('# CUR = "GHS"')

  AUTO-DETECTED SCHEMA
  Business Type : Fashion Retail
  Currency      : GHS

  customer    : Customer ID (Spoofed)
  date        : Purchase Date
  order       : Order ID (Hashed)
  revenue     : Total Line Amount
  price       : Unit Price
  qty         : Quantity
  discount    : Discount Amount
  promo       : Promo Code
  category    : Categories
  product     : Product Name
  payment     : Payment Method
  shipping    : Shipping Method

✅ Schema detection complete.

# ── MANUAL OVERRIDE (uncomment & edit if anything is wrong) ──
# SCHEMA["customer"] = "Customer ID (Spoofed)"
# SCHEMA["date"]     = "Purchase Date"
# SCHEMA["revenue"]  = "Total Line Amount"
# SCHEMA["order"]    = "Order ID (Hashed)"
# CUR = "GHS"


## Cell 4: Exploratory Data Analysis (EDA)

In [ ]:
print('=' * 60)
print('  EXPLORATORY DATA ANALYSIS')
print('=' * 60)

print(f'\n📊 Dataset Shape: {raw_df.shape}')
print(f'\n📈 Numerical Summary:\n{raw_df.describe().round(2)}')
print(f'\n❓ Missing Values:\n{raw_df.isnull().sum()[raw_df.isnull().sum() > 0]}')
print(f'\n🔁 Duplicate Rows: {raw_df.duplicated().sum()}')

# Date range
date_col = SCHEMA['date']
try:
    _dates = pd.to_datetime(raw_df[date_col], errors='coerce', utc=True).dropna()
    print(f'\n📅 Date Range: {_dates.min().date()} → {_dates.max().date()} ({(_dates.max()-_dates.min()).days} days)')
except Exception:
    pass

# Unique customers, products, categories
cust_col = SCHEMA['customer']
n_customers = raw_df[cust_col].nunique() if cust_col else 'N/A'
n_products  = raw_df[SCHEMA['product']].nunique() if SCHEMA['product'] else 'N/A'
n_cats      = raw_df[SCHEMA['category']].nunique() if SCHEMA['category'] else 'N/A'
print(f'\n👤 Unique Customers : {n_customers}')
print(f'📦 Unique Products   : {n_products}')
print(f'🏷️  Unique Categories : {n_cats}')

# Value counts for key categorical columns
for label, col in [('Status', SCHEMA['status']), ('Payment', SCHEMA['payment']),
                    ('Category', SCHEMA['category'])]:
    if col and col in raw_df.columns:
        print(f'\n{label} distribution:')
        print(raw_df[col].value_counts().head(8).to_string())

# Distribution histograms
rev_col = SCHEMA['revenue']
num_cols = [(c, l) for c, l in [(rev_col, 'Revenue per Transaction'),
             (SCHEMA['qty'], 'Quantity'), (SCHEMA['price'], 'Unit Price')]
            if c and c in raw_df.columns]
if num_cols:
    fig, axes = plt.subplots(1, len(num_cols), figsize=(6*len(num_cols), 4))
    if len(num_cols) == 1: axes = [axes]
    for ax, (col, label) in zip(axes, num_cols):
        data = pd.to_numeric(raw_df[col], errors='coerce').dropna()
        ax.hist(data, bins=40, color='#3498DB', edgecolor='white', alpha=0.8)
        ax.set_title(f'{label}\nSkew={data.skew():.2f}')
        ax.axvline(data.mean(), color='#E74C3C', linestyle='--', label=f'Mean: {data.mean():.1f}')
        ax.legend(fontsize=9)
    plt.suptitle('Raw Transaction Distributions', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/charts/eda_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

# Monthly transaction volume
try:
    _ts = pd.to_datetime(raw_df[date_col], errors='coerce', utc=True)
    monthly = _ts.dt.to_period('M').value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(range(len(monthly)), monthly.values, alpha=0.3, color='#3498DB')
    ax.plot(range(len(monthly)), monthly.values, color='#2C3E50', linewidth=1.5)
    ax.set_xticks(range(len(monthly)))
    ax.set_xticklabels([str(p) for p in monthly.index], rotation=45, ha='right', fontsize=8)
    ax.set_title('Monthly Transaction Volume', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/charts/eda_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'⚠️  Time series plot skipped: {e}')

# Top 10 products treemap
prod_col = SCHEMA['product']
if prod_col and prod_col in raw_df.columns:
    top_prods = raw_df[prod_col].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.Set3(np.linspace(0, 1, len(top_prods)))
    squarify.plot(sizes=top_prods.values,
                  label=[f'{str(n)[:20]}\n({v})' for n, v in zip(top_prods.index, top_prods.values)],
                  color=colors, alpha=0.85, text_kwargs={'fontsize': 8, 'fontweight': 'bold'}, ax=ax)
    ax.set_title('Top 10 Products by Transaction Count', fontsize=14, fontweight='bold')
    ax.axis('off'); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/charts/eda_treemap.png', dpi=150, bbox_inches='tight')
    plt.show()

print('\n✅ EDA complete')

  EXPLORATORY DATA ANALYSIS

📊 Dataset Shape: (1520, 17)

📈 Numerical Summary:
       Discount Amount  Quantity  Unit Price  Total Line Amount
count          1520.00   1520.00     1520.00            1520.00
mean              5.22      1.02      229.39             234.92
std              32.47      0.17      230.99             278.34
min               0.00      1.00        0.00               0.00
25%               0.00      1.00       80.00              80.00
50%               0.00      1.00      147.00             150.00
75%               0.00      1.00      321.00             321.00
max             510.00      4.00     2550.00            6000.00

❓ Missing Values:
Promo Code          1459
Customer Country       1
dtype: int64

🔁 Duplicate Rows: 0

📅 Date Range: 2025-03-06 → 2026-02-15 (346 days)

👤 Unique Customers : 795
📦 Unique Products   : 55
🏷️  Unique Categories : 11

Status distribution:
Order Status
completed     1389
confirmed       55
processing      53
pending         13
shi

## Cell 5: Data Cleaning

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: clean_data
# ══════════════════════════════════════════════
def clean_data(df, schema):
    """Clean and validate transaction data. Returns cleaned df and cleaning summary."""
    df = df.copy()
    report = {'original_rows': len(df)}

    # Remove exact duplicates
    before = len(df)
    df = df.drop_duplicates()
    report['duplicates_removed'] = before - len(df)

    # Parse date
    date_col = schema['date']
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    before = len(df)
    df = df.dropna(subset=[date_col])
    report['bad_dates_removed'] = before - len(df)

    # Revenue: coerce, fill from qty*price if possible, drop negatives/zeros
    rev_col = schema['revenue']
    df[rev_col] = pd.to_numeric(df[rev_col], errors='coerce')
    # Fill missing revenue from qty * price if both present
    if schema['qty'] and schema['price']:
        mask = df[rev_col].isna()
        if mask.any():
            df.loc[mask, rev_col] = (pd.to_numeric(df.loc[mask, schema['qty']], errors='coerce') *
                                     pd.to_numeric(df.loc[mask, schema['price']], errors='coerce'))
    before = len(df)
    df = df.dropna(subset=[rev_col])
    df = df[df[rev_col] > 0]  # remove returns, zero-value rows
    report['invalid_revenue_removed'] = before - len(df)

    # Filter valid order statuses if column present
    status_col = schema.get('status')
    if status_col and status_col in df.columns:
        valid = ['confirmed','delivered','shipped','completed','processing',
                 'fulfilled','paid','closed','success']
        mask = df[status_col].str.lower().str.strip().isin(valid)
        if mask.sum() > len(df) * 0.3:  # only filter if it keeps >30% of rows
            before = len(df)
            df = df[mask]
            report['invalid_status_removed'] = before - len(df)
        else:
            report['invalid_status_removed'] = 0  # skip filter, keep all

    # Fill missing categoricals with 'Unknown' / 'None'
    for col, default in [
        (schema.get('payment'),  'Unknown'),
        (schema.get('shipping'), 'Unknown'),
        (schema.get('promo'),    'None'),
        (schema.get('category'), 'Unknown'),
        (schema.get('product'),  'Unknown'),
    ]:
        if col and col in df.columns:
            df[col] = df[col].fillna(default)

    # Discount: coerce and fill NaN with 0
    disc_col = schema.get('discount')
    if disc_col and disc_col in df.columns:
        df[disc_col] = pd.to_numeric(df[disc_col], errors='coerce').fillna(0)

    report['final_rows'] = len(df)
    report['rows_removed'] = report['original_rows'] - report['final_rows']
    report['retention_pct'] = round(report['final_rows'] / report['original_rows'] * 100, 1)
    return df, report


print('=' * 60)
print('  DATA CLEANING')
print('=' * 60)

df, cleaning_report = clean_data(raw_df, SCHEMA)

print(f"  Original rows      : {cleaning_report['original_rows']:,}")
print(f"  Duplicates removed : {cleaning_report['duplicates_removed']:,}")
print(f"  Bad dates removed  : {cleaning_report['bad_dates_removed']:,}")
print(f"  Bad revenue removed: {cleaning_report['invalid_revenue_removed']:,}")
print(f"  Final rows         : {cleaning_report['final_rows']:,} ({cleaning_report['retention_pct']}% retained)")
print()
print('✅ Cleaning complete')

  DATA CLEANING
  Original rows      : 1,520
  Duplicates removed : 0
  Bad dates removed  : 0
  Bad revenue removed: 1
  Final rows         : 1,504 (98.9% retained)

✅ Cleaning complete


## Cell 6: Outlier Treatment (IQR Winsorization)

In [ ]:
print('=' * 60)
print('  OUTLIER TREATMENT (IQR Winsorization)')
print('=' * 60)

outlier_report = {}
cap_cols = [c for c in [SCHEMA['revenue'], SCHEMA['price'], SCHEMA['qty'], SCHEMA['discount']]
            if c and c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

if cap_cols:
    fig, axes = plt.subplots(2, len(cap_cols), figsize=(5*len(cap_cols), 8))
    if len(cap_cols) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, col in enumerate(cap_cols):
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = max(Q1 - 1.5*IQR, 0), Q3 + 1.5*IQR
        n_out = ((df[col] < lower) | (df[col] > upper)).sum()

        axes[0][i].hist(df[col], bins=40, color='#E74C3C', edgecolor='white', alpha=0.75)
        axes[0][i].set_title(f'{col}\nBEFORE — {n_out} outliers')
        axes[0][i].axvline(upper, color='black', linestyle='--', label=f'Cap: {upper:.0f}')
        axes[0][i].legend(fontsize=8)

        df[col] = df[col].clip(lower=lower, upper=upper)

        axes[1][i].hist(df[col], bins=40, color='#2ECC71', edgecolor='white', alpha=0.75)
        axes[1][i].set_title(f'{col}\nAFTER (Winsorized)')

        outlier_report[col] = {'n_capped': n_out, 'lower': round(lower, 2),
                                'upper': round(upper, 2), 'iqr': round(IQR, 2),
                                'pct': round(n_out/len(df)*100, 1)}
        print(f'  {col:<30}: {n_out} capped at [{lower:.2f}, {upper:.2f}]')

    plt.suptitle('Outlier Treatment — Before vs After', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/charts/outlier_treatment.png', dpi=150, bbox_inches='tight')
    plt.show()

print('\n✅ Outlier treatment complete')

  OUTLIER TREATMENT (IQR Winsorization)
  Total Line Amount             : 47 capped at [0.00, 630.00]
  Unit Price                    : 46 capped at [0.00, 631.50]
  Quantity                      : 29 capped at [1.00, 1.00]
  Discount Amount               : 71 capped at [0.00, 0.00]

✅ Outlier treatment complete


## Cell 7: RFM Feature Engineering ⭐

> **This is the core fix.** Instead of clustering raw transactions, we first aggregate to the **customer level** and compute Recency, Frequency, and Monetary value for each unique customer.

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: compute_rfm
# ══════════════════════════════════════════════
def compute_rfm(df, schema):
    """
    Aggregate transactions to customer level and compute RFM.
    Returns rfm_df with one row per customer: [CustomerID, Recency, Frequency, Monetary]
    """
    cust_col  = schema['customer']
    date_col  = schema['date']
    rev_col   = schema['revenue']
    order_col = schema.get('order')

    # reference date = day after last transaction
    reference_date = df[date_col].max() + timedelta(days=1)

    # Frequency = unique orders per customer if order_id exists, else transaction count
    if order_col and order_col in df.columns:
        freq_series = df.groupby(cust_col)[order_col].nunique()
    else:
        freq_series = df.groupby(cust_col)[rev_col].count()

    rfm = df.groupby(cust_col).agg(
        Recency  = (date_col,  lambda x: (reference_date - x.max()).days),
        Monetary = (rev_col,   'sum'),
    ).reset_index()

    rfm['Frequency'] = rfm[cust_col].map(freq_series)
    rfm = rfm.rename(columns={cust_col: 'CustomerID'})
    rfm = rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary']]

    # Drop customers with no valid spend or zero frequency
    before = len(rfm)
    rfm = rfm[(rfm['Monetary'] > 0) & (rfm['Frequency'] > 0)].reset_index(drop=True)
    dropped = before - len(rfm)

    return rfm, dropped


print('=' * 60)
print('  RFM FEATURE ENGINEERING (Customer Level)')
print('=' * 60)

rfm_df, rfm_dropped = compute_rfm(df, SCHEMA)

print(f'\n  📊 Aggregated {len(df):,} transactions → {len(rfm_df):,} unique customers')
if rfm_dropped > 0:
    print(f'  ⚠️  Dropped {rfm_dropped} customers with zero spend or frequency')
print()
print('  RFM Statistics:')
print(rfm_df[['Recency','Frequency','Monetary']].describe().round(2).to_string())

# Distribution plots — one per RFM feature
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#E74C3C', '#3498DB', '#2ECC71']
for ax, feat, color in zip(axes, ['Recency', 'Frequency', 'Monetary'], colors):
    ax.hist(rfm_df[feat], bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{feat}\nSkew={rfm_df[feat].skew():.2f}', fontsize=12, fontweight='bold')
    ax.axvline(rfm_df[feat].mean(), color='black', linestyle='--', alpha=0.7,
               label=f'Mean: {rfm_df[feat].mean():.1f}')
    ax.axvline(rfm_df[feat].median(), color='gray', linestyle=':', alpha=0.7,
               label=f'Median: {rfm_df[feat].median():.1f}')
    ax.legend(fontsize=9)
plt.suptitle(f'RFM Distributions — {len(rfm_df):,} Customers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ RFM computation complete')

  RFM FEATURE ENGINEERING (Customer Level)

  📊 Aggregated 1,504 transactions → 788 unique customers

  RFM Statistics:
       Recency  Frequency  Monetary
count   788.00      788.0    788.00
mean    242.57        1.0    401.86
std      66.00        0.0    403.44
min       1.00        1.0     50.00
25%     187.00        1.0    147.00
50%     273.00        1.0    289.00
75%     275.00        1.0    500.00
max     347.00        1.0   3379.79

✅ RFM computation complete


## Cell 8: Feature Scaling (Log Transform + StandardScaler)

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: scale_rfm
# ══════════════════════════════════════════════
def scale_rfm(rfm_df):
    """
    Log1p transform F and M (always); log1p R only if heavily right-skewed.
    Then StandardScaler on all three.
    Returns: scaled_data (numpy array), scaler, feature_labels.
    """
    data = rfm_df[['Recency', 'Frequency', 'Monetary']].copy()

    # Frequency and Monetary: always log-transform (highly right-skewed)
    data['Frequency'] = np.log1p(data['Frequency'])
    data['Monetary']  = np.log1p(data['Monetary'])

    # Recency: log-transform only if skew > 2
    r_skew = rfm_df['Recency'].skew()
    if r_skew > 2:
        data['Recency'] = np.log1p(data['Recency'])
        print(f'  ℹ️  Recency skew={r_skew:.2f} > 2 → log-transformed')
    else:
        print(f'  ℹ️  Recency skew={r_skew:.2f} ≤ 2 → left as-is')

    scaler = StandardScaler()
    scaled = scaler.fit_transform(data.values)
    return scaled, scaler, ['Recency', 'Frequency', 'Monetary']


print('=' * 60)
print('  FEATURE SCALING')
print('=' * 60)

scaled_data, scaler, FEAT_LABELS = scale_rfm(rfm_df)

print(f'\n  Scaled shape : {scaled_data.shape}  ({scaled_data.shape[0]} customers × {scaled_data.shape[1]} features)')
print(f'  Feature means (should be ≈ 0): {np.mean(scaled_data, axis=0).round(6)}')
print(f'  Feature stds  (should be ≈ 1): {np.std(scaled_data,  axis=0).round(6)}')
print('\n✅ Scaling complete')

  FEATURE SCALING
  ℹ️  Recency skew=-0.84 ≤ 2 → left as-is

  Scaled shape : (788, 3)  (788 customers × 3 features)
  Feature means (should be ≈ 0): [-0.  0.  0.]
  Feature stds  (should be ≈ 1): [1. 0. 1.]

✅ Scaling complete


In [ ]:
import pickle, json

# Save scaler artifact
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('💾 scaler.pkl saved')

💾 scaler.pkl saved


## Cell 9: PCA (Visualisation Only — clustering runs on 3D RFM space)

In [ ]:
print('=' * 60)
print('  PCA — FOR VISUALISATION ONLY')
print('  (Clustering will use the 3 RFM features, not PCA components)')
print('=' * 60)

pca_full = PCA()
pca_full.fit(scaled_data)
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

print('\n📊 Explained Variance per Component:')
for i, (ev, cv) in enumerate(zip(exp_var, cum_var)):
    marker = ' ← 85% threshold' if i > 0 and cum_var[i-1] < 0.85 and cv >= 0.85 else ''
    print(f'  PC{i+1}: {ev:.1%} (Cumulative: {cv:.1%}){marker}')

# 2D and 3D projections for scatter plots
n_2d = min(2, scaled_data.shape[1])
n_3d = min(3, scaled_data.shape[1])
pca   = PCA(n_components=n_2d)
pca_2d = pca.fit_transform(scaled_data)
pca3  = PCA(n_components=n_3d)
pca_3d = pca3.fit_transform(scaled_data)

print(f'\n  2D PCA captures {pca.explained_variance_ratio_.sum():.1%} of variance')
print(f'  3D PCA captures {pca3.explained_variance_ratio_.sum():.1%} of variance')

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(exp_var)+1), exp_var, color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].plot(range(1, len(exp_var)+1), exp_var, 'o-', color='#2C3E50', linewidth=2)
axes[0].set_title('Scree Plot'); axes[0].set_xlabel('PC'); axes[0].set_ylabel('Explained Variance')
axes[1].plot(range(1, len(cum_var)+1), cum_var, 's-', color='#E74C3C', linewidth=2.5, markersize=8)
axes[1].axhline(y=0.85, color='#2ECC71', linestyle='--', linewidth=2, label='85% Threshold')
axes[1].fill_between(range(1, len(cum_var)+1), cum_var, alpha=0.1, color='#E74C3C')
axes[1].set_title('Cumulative Variance'); axes[1].legend()
plt.suptitle('PCA Analysis (RFM Space)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

# Biplot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pca_2d[:, 0], pca_2d[:, 1], alpha=0.4, s=15, color='#95A5A6')
loadings = pca.components_.T
for i, feat in enumerate(FEAT_LABELS[:loadings.shape[0]]):
    ax.annotate('', xy=(loadings[i,0]*4, loadings[i,1]*4), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))
    ax.text(loadings[i,0]*4.3, loadings[i,1]*4.3, feat, fontsize=11, fontweight='bold', color='#C0392B')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA Biplot — RFM Feature Loadings', fontsize=13, fontweight='bold')
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ PCA complete')

  PCA — FOR VISUALISATION ONLY
  (Clustering will use the 3 RFM features, not PCA components)

📊 Explained Variance per Component:
  PC1: 53.4% (Cumulative: 53.4%)
  PC2: 46.6% (Cumulative: 100.0%) ← 85% threshold
  PC3: 0.0% (Cumulative: 100.0%)

  2D PCA captures 100.0% of variance
  3D PCA captures 100.0% of variance

✅ PCA complete


## Cell 10: Find Optimal K (Majority Vote across 3 metrics)

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: find_optimal_k
# ══════════════════════════════════════════════
def find_optimal_k(scaled_data, k_range=None):
    """
    Evaluate K=2..min(10, n-1), majority vote across Silhouette, CH, DB.
    Returns optimal_k, metrics_df.
    """
    n_samples = len(scaled_data)
    if k_range is None:
        k_range = range(2, min(11, n_samples))

    results = {'k': [], 'Inertia': [], 'Silhouette': [], 'Calinski-Harabasz': [], 'Davies-Bouldin': []}

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
        labs = km.fit_predict(scaled_data)
        results['k'].append(k)
        results['Inertia'].append(round(km.inertia_, 1))
        results['Silhouette'].append(round(silhouette_score(scaled_data, labs), 4))
        results['Calinski-Harabasz'].append(round(calinski_harabasz_score(scaled_data, labs), 1))
        results['Davies-Bouldin'].append(round(davies_bouldin_score(scaled_data, labs), 4))
        print(f'  k={k}: Sil={results["Silhouette"][-1]:.4f} | '
              f'CH={results["Calinski-Harabasz"][-1]:.1f} | '
              f'DB={results["Davies-Bouldin"][-1]:.4f}')

    best_sil = list(k_range)[np.argmax(results['Silhouette'])]
    best_ch  = list(k_range)[np.argmax(results['Calinski-Harabasz'])]
    best_db  = list(k_range)[np.argmin(results['Davies-Bouldin'])]

    votes = [best_sil, best_ch, best_db]
    vote_counts = Counter(votes)
    if vote_counts.most_common(1)[0][1] >= 2:
        optimal_k = vote_counts.most_common(1)[0][0]
        reason = 'majority vote (2+ metrics agree)'
    else:
        optimal_k = best_sil
        reason = 'tiebreaker: best Silhouette'

    return optimal_k, pd.DataFrame(results), reason, (best_sil, best_ch, best_db)


print('=' * 60)
print('  OPTIMAL K SELECTION')
print('=' * 60)

optimal_k, metrics_df, k_reason, (best_sil_k, best_ch_k, best_db_k) = find_optimal_k(scaled_data)

print(f'\n  Best by Silhouette      : k={best_sil_k}')
print(f'  Best by Calinski-H      : k={best_ch_k}')
print(f'  Best by Davies-Bouldin  : k={best_db_k}')
print(f'\n  ✅ OPTIMAL k = {optimal_k}  ({k_reason})')

# 4-panel metric plots
k_range = metrics_df['k'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_cfg = [('Inertia','#3498DB','lower'),('Silhouette','#2ECC71','higher'),
              ('Calinski-Harabasz','#F39C12','higher'),('Davies-Bouldin','#E74C3C','lower')]
for ax, (metric, color, direction) in zip(axes.flat, metric_cfg):
    vals = metrics_df[metric].tolist()
    ax.plot(k_range, vals, 'o-', color=color, linewidth=2.5, markersize=8)
    ax.fill_between(k_range, vals, alpha=0.1, color=color)
    ax.axvline(x=optimal_k, color='grey', linestyle=':', linewidth=2, alpha=0.7,
               label=f'Chosen k={optimal_k}')
    best_idx = np.argmax(vals) if direction == 'higher' else np.argmin(vals)
    ax.scatter([k_range[best_idx]], [vals[best_idx]], s=200, color=color,
               zorder=5, edgecolors='black', linewidths=2)
    ax.set_title(f'{metric} ({direction} = better)', fontsize=12, fontweight='bold')
    ax.set_xlabel('k'); ax.set_ylabel(metric); ax.legend()
plt.suptitle(f'K Selection Metrics — Chosen k={optimal_k}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/optimal_k_4metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Silhouette plot for chosen k
km_sil = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
_labels_sil = km_sil.fit_predict(scaled_data)
sil_vals = silhouette_samples(scaled_data, _labels_sil)
fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
colors_sil = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    clust = np.sort(sil_vals[_labels_sil == i])
    y_upper = y_lower + len(clust)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, clust, alpha=0.7, color=colors_sil[i])
    ax.text(-0.05, y_lower + 0.5*len(clust), f'Cluster {i}', fontsize=10, fontweight='bold')
    y_lower = y_upper + 10
avg_sil = metrics_df.loc[metrics_df['k']==optimal_k, 'Silhouette'].values[0]
ax.axvline(x=avg_sil, color='red', linestyle='--', lw=2, label=f'Avg: {avg_sil:.3f}')
ax.set_title(f'Silhouette Plot (k={optimal_k})', fontsize=14, fontweight='bold')
ax.set_xlabel('Silhouette Coefficient'); ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Optimal k = {optimal_k}')

  OPTIMAL K SELECTION
  k=2: Sil=0.3898 | CH=449.2 | DB=1.1243
  k=3: Sil=0.4158 | CH=642.0 | DB=0.8252
  k=4: Sil=0.4146 | CH=659.3 | DB=0.7999
  k=5: Sil=0.4245 | CH=662.6 | DB=0.7746
  k=6: Sil=0.4101 | CH=724.3 | DB=0.8043
  k=7: Sil=0.3956 | CH=701.2 | DB=0.8247
  k=8: Sil=0.3867 | CH=697.4 | DB=0.8682
  k=9: Sil=0.4089 | CH=716.5 | DB=0.7915
  k=10: Sil=0.4162 | CH=708.5 | DB=0.8153

  Best by Silhouette      : k=5
  Best by Calinski-H      : k=6
  Best by Davies-Bouldin  : k=5

  ✅ OPTIMAL k = 5  (majority vote (2+ metrics agree))

✅ Optimal k = 5


## Cell 11: Run THREE Clustering Algorithms + Pick Best ⭐

> Runs K-Means, GMM, and Hierarchical Clustering. Compares all three by Silhouette Score. The best algorithm's labels are used for all downstream work.

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: run_all_clustering
# ══════════════════════════════════════════════
def run_all_clustering(scaled_data, optimal_k):
    """
    Run K-Means, GMM, Hierarchical. Compare by Silhouette. Return best labels.
    Returns: best_labels, best_algo_name, comparison_dict.
    """
    results = {}

    # ── K-Means ──────────────────────────────────────────
    km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
    km_labels = km.fit_predict(scaled_data)
    km_sil = silhouette_score(scaled_data, km_labels)
    km_db  = davies_bouldin_score(scaled_data, km_labels)
    km_ch  = calinski_harabasz_score(scaled_data, km_labels)
    results['K-Means'] = {
        'labels': km_labels, 'model': km,
        'Silhouette': round(km_sil, 4), 'Davies-Bouldin': round(km_db, 4), 'Calinski-H': round(km_ch, 1)
    }

    # ── Gaussian Mixture Model ───────────────────────────
    gmm = GaussianMixture(n_components=optimal_k, covariance_type='full',
                           random_state=42, n_init=3)
    gmm.fit(scaled_data)
    gmm_labels = gmm.predict(scaled_data)  # hard assignment
    gmm_sil = silhouette_score(scaled_data, gmm_labels)
    gmm_db  = davies_bouldin_score(scaled_data, gmm_labels)
    gmm_ch  = calinski_harabasz_score(scaled_data, gmm_labels)
    results['GMM'] = {
        'labels': gmm_labels, 'model': gmm,
        'Silhouette': round(gmm_sil, 4), 'Davies-Bouldin': round(gmm_db, 4), 'Calinski-H': round(gmm_ch, 1)
    }

    # ── Hierarchical (Agglomerative, Ward linkage) ───────
    hc = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
    hc_labels = hc.fit_predict(scaled_data)
    hc_sil = silhouette_score(scaled_data, hc_labels)
    hc_db  = davies_bouldin_score(scaled_data, hc_labels)
    hc_ch  = calinski_harabasz_score(scaled_data, hc_labels)
    results['Hierarchical'] = {
        'labels': hc_labels, 'model': hc,
        'Silhouette': round(hc_sil, 4), 'Davies-Bouldin': round(hc_db, 4), 'Calinski-H': round(hc_ch, 1)
    }

    # ── Pick best by highest Silhouette Score ─────────────
    best_algo = max(results.items(), key=lambda x: x[1]['Silhouette'])
    best_name   = best_algo[0]
    best_labels = best_algo[1]['labels']

    return best_labels, best_name, results


print('=' * 60)
print(f'  THREE CLUSTERING ALGORITHMS (k={optimal_k})')
print('=' * 60)

cluster_labels, best_algo_name, algo_comparison = run_all_clustering(scaled_data, optimal_k)

# Print comparison table
print(f'\n  {"Algorithm":<16} {"Silhouette":>12} {"Davies-Bouldin":>16} {"Calinski-H":>12}')
print(f'  {"-"*58}')
for algo, res in algo_comparison.items():
    marker = '  ← BEST' if algo == best_algo_name else ''
    print(f'  {algo:<16} {res["Silhouette"]:>12.4f} {res["Davies-Bouldin"]:>16.4f} {res["Calinski-H"]:>12.1f}{marker}')

best_res = algo_comparison[best_algo_name]
print(f'\n  ✅ Best algorithm: {best_algo_name}  (Silhouette = {best_res["Silhouette"]:.4f})')
print(f'  All downstream analysis uses {best_algo_name} labels.')

# Attach best labels to rfm_df
rfm_df['Cluster'] = cluster_labels
print(f'\n  Cluster sizes:')
for c, n in zip(*np.unique(cluster_labels, return_counts=True)):
    print(f'    Cluster {c}: {n} customers ({n/len(cluster_labels)*100:.1f}%)')

# Store final metrics
final_sil = best_res['Silhouette']
final_db  = best_res['Davies-Bouldin']
final_ch  = best_res['Calinski-H']

# Algorithm comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
algo_names = list(algo_comparison.keys())
pal = ['#3498DB' if a != best_algo_name else '#2ECC71' for a in algo_names]
for ax, metric, direction in zip(axes, ['Silhouette','Davies-Bouldin','Calinski-H'],
                                        ['higher=better','lower=better','higher=better']):
    vals = [algo_comparison[a][metric] for a in algo_names]
    bars = ax.bar(algo_names, vals, color=pal, edgecolor='white', linewidth=2)
    ax.set_title(f'{metric}\n({direction})', fontsize=12, fontweight='bold')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.suptitle(f'Algorithm Comparison — Winner: {best_algo_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

  THREE CLUSTERING ALGORITHMS (k=5)

  Algorithm          Silhouette   Davies-Bouldin   Calinski-H
  ----------------------------------------------------------
  K-Means                0.4245           0.7746        662.6  ← BEST
  GMM                    0.1149           1.6725        225.5
  Hierarchical           0.3745           0.8019        581.6

  ✅ Best algorithm: K-Means  (Silhouette = 0.4245)
  All downstream analysis uses K-Means labels.

  Cluster sizes:
    Cluster 0: 210 customers (26.6%)
    Cluster 1: 132 customers (16.8%)
    Cluster 2: 274 customers (34.8%)
    Cluster 3: 42 customers (5.3%)
    Cluster 4: 130 customers (16.5%)


## Cell 12: Visualise Clusters in PCA Space

In [ ]:
print('=' * 60)
print('  CLUSTER VISUALISATION (PCA Projection)')
print('=' * 60)

colors_cluster = plt.cm.Set2(np.linspace(0, 1, optimal_k))

fig, ax = plt.subplots(figsize=(12, 8))
for i in range(optimal_k):
    mask = cluster_labels == i
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1], c=[colors_cluster[i]],
               label=f'Cluster {i} (n={mask.sum()})', alpha=0.6, s=50,
               edgecolors='white', linewidth=0.4)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title(f'{best_algo_name} Clusters in PCA Space (k={optimal_k})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/pca_clusters_2d.png', dpi=150, bbox_inches='tight')
plt.show()

# 3D interactive
pca_df_3d = pd.DataFrame({'PC1': pca_3d[:,0], 'PC2': pca_3d[:,1], 'PC3': pca_3d[:,2],
                           'Cluster': [f'Cluster {c}' for c in cluster_labels]})
fig3d = px.scatter_3d(pca_df_3d, x='PC1', y='PC2', z='PC3', color='Cluster',
                       opacity=0.6, title=f'{best_algo_name} — 3D PCA Cluster View')
fig3d.update_layout(height=600)
fig3d.show()

print('✅ Cluster visualisation complete')

  CLUSTER VISUALISATION (PCA Projection)


✅ Cluster visualisation complete


## Cell 13: RFM Segment Labelling ⭐

> Assigns industry-standard segment names (Champions, Loyal, At Risk, etc.) using RFM percentile thresholds — not spending tiers.

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: label_segments
# ══════════════════════════════════════════════
def label_segments(rfm_df, cluster_labels):
    """
    Assign RFM-based segment names using percentile thresholds.
    Compute per-cluster median R, F, M and map to canonical segment names.
    Returns: segment_map (cluster_id -> name), labelled_rfm_df.
    """
    rfm = rfm_df.copy()
    rfm['Cluster'] = cluster_labels

    # RFM percentile scoring (1–4 per feature)
    # Recency: LOWER is better (bought recently) → score 4 if small R
    # Frequency and Monetary: HIGHER is better
    def _safe_qcut(series, ascending=True):
        """qcut with fallback if duplicates collapse the bins below 4."""
        for q in [4, 3, 2]:
            try:
                n_labels = q
                if ascending:
                    labels = list(range(1, n_labels + 1))
                else:
                    labels = list(range(n_labels, 0, -1))
                result = pd.qcut(series, q=q, labels=labels, duplicates='drop')
                return result.astype(float)
            except ValueError:
                continue
        # last resort: rank-based scoring 1-4
        return pd.cut(series, bins=4, labels=[1,2,3,4] if ascending else [4,3,2,1],
                      duplicates='drop').astype(float)

    rfm['R_score'] = _safe_qcut(rfm['Recency'],   ascending=False)  # lower recency = better
    rfm['F_score'] = _safe_qcut(rfm['Frequency'],  ascending=True)
    rfm['M_score'] = _safe_qcut(rfm['Monetary'],   ascending=True)
    rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].fillna(2.0)

    # Cluster-level median scores
    cluster_scores = rfm.groupby('Cluster')[['R_score','F_score','M_score']].median()
    cluster_scores['RFM_total'] = cluster_scores.sum(axis=1)

    def _assign_name(row):
        r, f, m = row['R_score'], row['F_score'], row['M_score']
        total = row['RFM_total']
        # Map to canonical RFM segment names
        if r >= 3.5 and f >= 3.5 and m >= 3.5:
            return 'Champions'
        elif r >= 3 and f >= 3 and m >= 3:
            return 'Loyal Customers'
        elif r >= 3 and f >= 2 and m >= 2:
            return 'Potential Loyalists'
        elif r >= 3.5 and f <= 1.5:
            return 'New Customers'
        elif r >= 2.5 and f >= 1.5 and m >= 1.5:
            return 'Promising'
        elif r >= 2 and f >= 2 and m >= 2:
            return 'Need Attention'
        elif r >= 2 and f <= 2 and m <= 2:
            return 'About to Sleep'
        elif r <= 2 and f >= 3 and m >= 3:
            return 'At Risk'
        elif r <= 1.5 and f >= 3.5 and m >= 3.5:
            return 'Cannot Lose Them'
        elif r <= 2 and f <= 2:
            return 'Hibernating'
        else:
            return 'Lost Customers'

    cluster_scores['SegmentName'] = cluster_scores.apply(_assign_name, axis=1)

    # Handle duplicate names: append cluster id suffix if two clusters map to same name
    seen = Counter()
    final_names = {}
    for cid in cluster_scores.index:
        name = cluster_scores.loc[cid, 'SegmentName']
        seen[name] += 1
        if seen[name] > 1:
            final_names[cid] = f'{name} ({seen[name]})'
        else:
            final_names[cid] = name
    # Fix earlier entries that now need suffix
    for name, count in seen.items():
        if count > 1:
            first_cid = [c for c, n in final_names.items() if n == name][0]
            final_names[first_cid] = f'{name} (1)'

    segment_map = final_names  # {cluster_id: segment_name}
    rfm['Segment'] = rfm['Cluster'].map(segment_map)

    return segment_map, rfm, cluster_scores


print('=' * 60)
print('  RFM SEGMENT LABELLING')
print('=' * 60)

segment_map, rfm_labelled, cluster_rfm_scores = label_segments(rfm_df, cluster_labels)
rfm_df = rfm_labelled
segments = list(segment_map.values())

PALETTE = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22','#34495E']
seg_color_map = {seg: PALETTE[i % len(PALETTE)] for i, seg in enumerate(segments)}

print('\n  Cluster → Segment Mapping:')
print(f'  {"Cluster":<10} {"Segment":<25} {"R_score":<10} {"F_score":<10} {"M_score":<10} {"Count"}')
print(f'  {"-"*72}')
for cid in sorted(segment_map.keys()):
    seg  = segment_map[cid]
    row  = cluster_rfm_scores.loc[cid]
    cnt  = (cluster_labels == cid).sum()
    print(f'  {cid:<10} {seg:<25} {row["R_score"]:<10.1f} {row["F_score"]:<10.1f} {row["M_score"]:<10.1f} {cnt}')

print('\n✅ Segment labelling complete')

  RFM SEGMENT LABELLING

  Cluster → Segment Mapping:
  Cluster    Segment                   R_score    F_score    M_score    Count
  ------------------------------------------------------------------------
  0          About to Sleep (1)        2.0        2.0        1.0        210
  1          Potential Loyalists (1)   4.0        2.0        3.0        132
  2          Hibernating               1.0        2.0        3.0        274
  3          Potential Loyalists (2)   4.0        2.0        4.0        42
  4          About to Sleep (2)        4.0        2.0        1.0        130

✅ Segment labelling complete


In [ ]:
# Save segment map and cluster RFM score summary
with open(f'{OUTPUT_DIR}/segment_map.json', 'w') as f:
    json.dump({str(k): v for k, v in segment_map.items()}, f, indent=2)

cluster_rfm_scores.to_csv(f'{OUTPUT_DIR}/cluster_rfm_profile.csv')
print('💾 segment_map.json saved')
print('💾 cluster_rfm_profile.csv saved')

💾 segment_map.json saved
💾 cluster_rfm_profile.csv saved


## Cell 14: Segment Profiling — Join back to Transaction Data

In [ ]:
print('=' * 60)
print('  SEGMENT PROFILING')
print('=' * 60)

# Join segment labels back to transaction-level df for behavioural profiling
cust_col = SCHEMA['customer']
df = df.merge(
    rfm_df[['CustomerID', 'Cluster', 'Segment', 'Recency', 'Frequency', 'Monetary']],
    left_on=cust_col, right_on='CustomerID', how='left'
)

def safe_mode(series):
    m = series.dropna().mode()
    return str(m.iloc[0]) if len(m) > 0 else 'N/A'

segment_insights = {}
for seg in segments:
    # Customer-level stats come from rfm_df
    cust_seg = rfm_df[rfm_df['Segment'] == seg]
    # Transaction-level stats come from df
    txn_seg  = df[df['Segment'] == seg]
    if len(cust_seg) == 0:
        continue

    info = {
        'customer_count':    len(cust_seg),
        'customer_pct':      round(len(cust_seg)/len(rfm_df)*100, 1),
        'transaction_count': len(txn_seg),
        'avg_recency':       round(cust_seg['Recency'].mean(), 1),
        'avg_frequency':     round(cust_seg['Frequency'].mean(), 1),
        'avg_monetary':      round(cust_seg['Monetary'].mean(), 2),
        'median_monetary':   round(cust_seg['Monetary'].median(), 2),
        'total_revenue':     round(cust_seg['Monetary'].sum(), 2),
    }

    # Behavioural enrichment from transactions
    rev_col  = SCHEMA['revenue']
    disc_col = SCHEMA['discount']
    promo_col= SCHEMA['promo']
    info['avg_spending']    = round(txn_seg[rev_col].mean(), 2) if rev_col and len(txn_seg) > 0 else info['avg_monetary']
    info['avg_discount']    = round(txn_seg[disc_col].mean(), 2) if disc_col and disc_col in df.columns and len(txn_seg) > 0 else 0
    info['discount_rate']   = round((txn_seg[disc_col] > 0).mean()*100, 1) if disc_col and disc_col in df.columns and len(txn_seg) > 0 else 0

    if promo_col and promo_col in df.columns and len(txn_seg) > 0:
        promo_none = ['none','nan','','null','no','0','n/a','na']
        info['promo_rate'] = round((~txn_seg[promo_col].astype(str).str.lower().str.strip().isin(promo_none)).mean()*100, 1)
    else:
        info['promo_rate'] = 0

    info['top_category'] = safe_mode(txn_seg[SCHEMA['category']]) if SCHEMA['category'] and SCHEMA['category'] in df.columns and len(txn_seg) > 0 else 'N/A'
    info['top_product']  = safe_mode(txn_seg[SCHEMA['product']])  if SCHEMA['product'] and SCHEMA['product'] in df.columns and len(txn_seg) > 0 else 'N/A'
    info['top_payment']  = safe_mode(txn_seg[SCHEMA['payment']])  if SCHEMA['payment'] and SCHEMA['payment'] in df.columns and len(txn_seg) > 0 else 'N/A'

    segment_insights[seg] = info

    print(f'\n  🎯 {seg}')
    print(f'     Customers  : {info["customer_count"]} ({info["customer_pct"]}%)')
    print(f'     Avg R/F/M  : {info["avg_recency"]} days / {info["avg_frequency"]} orders / {CUR} {info["avg_monetary"]:,.2f}')
    print(f'     Revenue    : {CUR} {info["total_revenue"]:,.2f} total  |  {CUR} {info["avg_monetary"]:,.2f} avg per customer')
    print(f'     Discount   : {info["discount_rate"]}%  |  Promo: {info["promo_rate"]}%')

print('\n✅ Profiling complete')

  SEGMENT PROFILING

  🎯 About to Sleep (1)
     Customers  : 210 (26.6%)
     Avg R/F/M  : 278.3 days / 1.0 orders / GHS 139.95
     Revenue    : GHS 29,388.73 total  |  GHS 139.95 avg per customer
     Discount   : 0.0%  |  Promo: 1.0%

  🎯 Potential Loyalists (1)
     Customers  : 132 (16.8%)
     Avg R/F/M  : 187.4 days / 1.0 orders / GHS 517.70
     Revenue    : GHS 68,336.00 total  |  GHS 517.70 avg per customer
     Discount   : 0.0%  |  Promo: 12.4%

  🎯 Hibernating
     Customers  : 274 (34.8%)
     Avg R/F/M  : 293.6 days / 1.0 orders / GHS 560.38
     Revenue    : GHS 153,543.82 total  |  GHS 560.38 avg per customer
     Discount   : 0.0%  |  Promo: 2.7%

  🎯 Potential Loyalists (2)
     Customers  : 42 (5.3%)
     Avg R/F/M  : 84.0 days / 1.0 orders / GHS 1,154.17
     Revenue    : GHS 48,475.00 total  |  GHS 1,154.17 avg per customer
     Discount   : 0.0%  |  Promo: 2.6%

  🎯 About to Sleep (2)
     Customers  : 130 (16.5%)
     Avg R/F/M  : 184.6 days / 1.0 orders / GHS 

## Cell 15: RFM Violin Plots & Radar Charts

In [ ]:
print('=' * 60)
print('  RFM DISTRIBUTION & RADAR CHARTS')
print('=' * 60)

seg_colors = [seg_color_map[s] for s in segments]

# RFM violin plots — one per R/F/M
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, feat, color, label in zip(axes,
    ['Recency', 'Frequency', 'Monetary'],
    ['#E74C3C', '#3498DB', '#2ECC71'],
    [f'Recency (days)', f'Frequency (orders)', f'Monetary ({CUR})']):

    data_list = [rfm_df[rfm_df['Segment']==s][feat].dropna().values for s in segments]
    data_list = [d if len(d) > 0 else np.array([0]) for d in data_list]
    vp = ax.violinplot(data_list, positions=range(len(segments)), showmeans=True, showmedians=True)
    for j, body in enumerate(vp['bodies']):
        body.set_facecolor(seg_colors[j]); body.set_alpha(0.6)
    vp['cmeans'].set_color('red'); vp['cmedians'].set_color('black')
    ax.set_xticks(range(len(segments)))
    ax.set_xticklabels(segments, rotation=25, ha='right', fontsize=9)
    ax.set_title(label, fontsize=12, fontweight='bold')

plt.suptitle('RFM Distribution by Segment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/rfm_violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# Radar chart
radar_feats = ['Recency', 'Frequency', 'Monetary']
radar_data  = rfm_df.groupby('Segment')[radar_feats].median()
# Invert Recency: lower = better, so we invert for radar so outward = better
radar_data['Recency'] = radar_data['Recency'].max() - radar_data['Recency'] + 1
radar_norm  = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-10)

angles = np.linspace(0, 2*np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for i, seg in enumerate(segments):
    if seg not in radar_norm.index:
        continue
    vals = radar_norm.loc[seg].values.tolist() + [radar_norm.loc[seg].values[0]]
    color = seg_color_map.get(seg, PALETTE[i % len(PALETTE)])
    ax.fill(angles, vals, alpha=0.15, color=color)
    ax.plot(angles, vals, 'o-', linewidth=2, markersize=6, label=seg, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Recency (inv.)','Frequency','Monetary'], fontsize=12, fontweight='bold')
ax.set_title('RFM Segment Radar Chart', fontsize=14, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Visualisations complete')

  RFM DISTRIBUTION & RADAR CHARTS
✅ Visualisations complete


## Cell 16: Behavioural Analysis Charts

In [ ]:
print('=' * 60)
print('  BEHAVIOURAL ANALYSIS')
print('=' * 60)

total_revenue = sum(info['total_revenue'] for info in segment_insights.values())

# Customer count bar + pie
cust_counts = [segment_insights.get(s, {}).get('customer_count', 0) for s in segments]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
bars = axes[0].bar(segments, cust_counts, color=seg_colors, edgecolor='white', linewidth=2)
axes[0].set_title('Customer Count by Segment', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, cust_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[1].pie(cust_counts, labels=segments, colors=seg_colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Customer Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Revenue bar + avg monetary + category + payment
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

rev_vals = [segment_insights.get(s, {}).get('total_revenue', 0) for s in segments]
bars = axes[0][0].bar(segments, rev_vals, color=seg_colors, edgecolor='white')
axes[0][0].set_title(f'Total Revenue per Segment ({CUR})', fontsize=12, fontweight='bold')
axes[0][0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, rev_vals):
    axes[0][0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(rev_vals)*0.01,
                    f'{val:,.0f}', ha='center', fontsize=8, fontweight='bold')

avg_mon = [segment_insights.get(s, {}).get('avg_monetary', 0) for s in segments]
bars = axes[0][1].bar(segments, avg_mon, color=seg_colors, edgecolor='white')
axes[0][1].set_title(f'Avg Monetary per Customer ({CUR})', fontsize=12, fontweight='bold')
axes[0][1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, avg_mon):
    axes[0][1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(avg_mon)*0.01,
                    f'{CUR} {val:,.0f}', ha='center', fontsize=8, fontweight='bold')

if SCHEMA['category'] and SCHEMA['category'] in df.columns:
    cat_cross = pd.crosstab(df['Segment'], df[SCHEMA['category']], normalize='index')
    top_cats = cat_cross.sum().nlargest(6).index
    cat_cross[top_cats].reindex(segments).plot(kind='bar', stacked=True, ax=axes[1][0],
                                                colormap='Set2', edgecolor='white')
    axes[1][0].set_title('Category Preferences by Segment', fontsize=12, fontweight='bold')
    axes[1][0].legend(fontsize=7, loc='upper right')
    axes[1][0].tick_params(axis='x', rotation=20)

if SCHEMA['payment'] and SCHEMA['payment'] in df.columns:
    pay_cross = pd.crosstab(df['Segment'], df[SCHEMA['payment']], normalize='index').reindex(segments)
    pay_cross.plot(kind='bar', stacked=True, ax=axes[1][1], colormap='Pastel1', edgecolor='white')
    axes[1][1].set_title('Payment Methods by Segment', fontsize=12, fontweight='bold')
    axes[1][1].legend(fontsize=7)
    axes[1][1].tick_params(axis='x', rotation=20)

plt.suptitle('Behavioural Analysis by Segment', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/behavioral_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Behavioural analysis complete')

  BEHAVIOURAL ANALYSIS
✅ Behavioural analysis complete


## Cell 17: Statistical Validation (ANOVA on RFM + ARI Stability)

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: validate_clusters
# ══════════════════════════════════════════════
def validate_clusters(scaled_data, cluster_labels, rfm_df, optimal_k):
    """
    ANOVA per RFM feature across clusters + ARI stability over 10 runs.
    Returns validation_results dict.
    """
    results = {'anova': {}, 'ari': {}}

    # ANOVA on actual RFM features (not raw transaction columns)
    for feat in ['Recency', 'Frequency', 'Monetary']:
        groups = [rfm_df[rfm_df['Cluster']==c][feat].dropna().values
                  for c in rfm_df['Cluster'].unique()]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) >= 2:
            try:
                f_stat, p_val = stats.f_oneway(*groups)
                results['anova'][feat] = {
                    'F': round(float(f_stat), 2),
                    'p': float(p_val),
                    'sig': p_val < 0.05
                }
            except Exception:
                pass

    # ARI stability over 10 KMeans runs
    reference = KMeans(n_clusters=optimal_k, random_state=42, n_init=10).fit_predict(scaled_data)
    ari_scores = []
    for seed in range(1, 11):
        labs_i = KMeans(n_clusters=optimal_k, random_state=seed*7, n_init=10).fit_predict(scaled_data)
        ari_scores.append(round(adjusted_rand_score(reference, labs_i), 4))

    avg_ari = np.mean(ari_scores)
    std_ari = np.std(ari_scores)
    stability = ('Excellent' if avg_ari > 0.9 else
                 'Good' if avg_ari > 0.7 else
                 'Fair' if avg_ari > 0.5 else 'Poor')
    results['ari'] = {
        'scores': ari_scores, 'avg': round(avg_ari, 4),
        'std': round(std_ari, 4), 'stability': stability
    }
    return results


print('=' * 60)
print('  STATISTICAL VALIDATION')
print('=' * 60)

validation_results = validate_clusters(scaled_data, cluster_labels, rfm_df, optimal_k)

# ANOVA results — on RFM features
print('\n  ANOVA Tests (on RFM features):')
print(f'  {"Feature":<15} {"F-stat":>10} {"p-value":>14} {"Significant?":>14}')
print(f'  {"-"*55}')
anova_results = validation_results['anova']
for feat, res in anova_results.items():
    sig = 'Yes (p<0.05)' if res['sig'] else 'No'
    print(f'  {feat:<15} {res["F"]:>10.2f} {res["p"]:>14.2e} {sig:>14}')

# ARI stability
ari = validation_results['ari']
avg_ari = ari['avg']
std_ari = ari['std']
stability = ari['stability']
n_runs = len(ari['scores'])

print(f'\n  ARI Stability ({n_runs} runs):')
for i, score in enumerate(ari['scores'], 1):
    print(f'    Run {i:>2d}: ARI = {score:.4f}')
print(f'\n  Avg ARI: {avg_ari:.4f} ± {std_ari:.4f} → {stability}')

# ARI chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars_ari = axes[0].bar(range(1, n_runs+1), ari['scores'], color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].axhline(y=avg_ari, color='#E74C3C', linestyle='--', lw=2, label=f'Mean ARI: {avg_ari:.4f}')
axes[0].set_title(f'Cluster Stability ({n_runs} runs) — {stability}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Run'); axes[0].set_ylabel('Adjusted Rand Index')
axes[0].set_ylim(0, 1.05); axes[0].legend()

# ANOVA F-stats
if anova_results:
    feats_a = list(anova_results.keys())
    f_vals  = [anova_results[f]['F'] for f in feats_a]
    a_colors = ['#2ECC71' if anova_results[f]['sig'] else '#E74C3C' for f in feats_a]
    bars_a = axes[1].bar(feats_a, f_vals, color=a_colors, edgecolor='white')
    axes[1].set_title('ANOVA F-Statistics (RFM)\nGreen = Significant (p<0.05)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('F-statistic')
    for bar, val, feat in zip(bars_a, f_vals, feats_a):
        p = anova_results[feat]['p']
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(f_vals)*0.02,
                     f'F={val:.1f}\np={p:.1e}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Statistical Validation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/statistical_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Statistical validation complete')

  STATISTICAL VALIDATION

  ANOVA Tests (on RFM features):
  Feature             F-stat        p-value   Significant?
  -------------------------------------------------------
  Recency            1016.20      3.42e-308   Yes (p<0.05)
  Frequency              nan            nan             No
  Monetary            154.13       3.12e-97   Yes (p<0.05)

  ARI Stability (10 runs):
    Run  1: ARI = 1.0000
    Run  2: ARI = 1.0000
    Run  3: ARI = 1.0000
    Run  4: ARI = 1.0000
    Run  5: ARI = 1.0000
    Run  6: ARI = 1.0000
    Run  7: ARI = 1.0000
    Run  8: ARI = 1.0000
    Run  9: ARI = 1.0000
    Run 10: ARI = 1.0000

  Avg ARI: 1.0000 ± 0.0000 → Excellent

✅ Statistical validation complete


## Cell 18: Explainable AI (SHAP — on RFM customer vectors)

In [ ]:
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: explain_with_shap
# ══════════════════════════════════════════════
# ══════════════════════════════════════════════
# SECTION A — CORE FUNCTION: explain_with_shap
# ══════════════════════════════════════════════
def explain_with_shap(scaled_data, cluster_labels, feature_labels):
    """
    Train Random Forest surrogate on customer-level RFM vectors.
    Apply SHAP TreeExplainer. Return shap_values, importances, surrogate_accuracy.
    """
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    rf.fit(scaled_data, cluster_labels)
    acc = rf.score(scaled_data, cluster_labels)

    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(scaled_data)

    # Handle all SHAP output shapes across versions:
    # - Old SHAP: list of (n_samples, n_features) arrays, one per class
    # - New SHAP: single ndarray of shape (n_samples, n_features, n_classes)
    #             OR (n_classes, n_samples, n_features)
    sv = shap_values
    if isinstance(sv, list):
        # Old format — list of 2D arrays
        mean_abs = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        # New format — figure out which axis is n_classes
        # axis 0 = n_classes if sv.shape[0] == n_classes, else axis 2
        n_classes = len(np.unique(cluster_labels))
        if sv.shape[0] == n_classes:
            # shape: (n_classes, n_samples, n_features)
            mean_abs = np.mean(np.abs(sv), axis=(0, 1))
        else:
            # shape: (n_samples, n_features, n_classes)
            mean_abs = np.mean(np.abs(sv), axis=(0, 2))
    else:
        # 2D fallback — binary or already collapsed
        mean_abs = np.abs(sv).mean(axis=0)

    # Guarantee 1D array of length == n_features before building dict
    mean_abs = np.array(mean_abs).flatten()
    if len(mean_abs) != len(feature_labels):
        # Last resort: use RF's built-in feature importances instead
        mean_abs = rf.feature_importances_

    importances = {feat: round(float(val), 6)
                   for feat, val in zip(feature_labels, mean_abs)}
    return shap_values, importances, round(acc, 4)


print('=' * 60)
print('  EXPLAINABLE AI (SHAP) — on customer RFM vectors')
print('=' * 60)

shap_values, shap_importances, surrogate_acc = explain_with_shap(scaled_data, cluster_labels, FEAT_LABELS)
combined_imp = np.array(list(shap_importances.values()))
combined_imp = combined_imp / (combined_imp.sum() + 1e-10)  # normalise to sum=1

print(f'\n  Surrogate RF accuracy: {surrogate_acc:.1%}')
print('\n  Feature Importance (mean |SHAP|):')
for feat, score in sorted(zip(FEAT_LABELS, combined_imp), key=lambda x: x[1], reverse=True):
    bar = '█' * int(score * 40)
    print(f'    {feat:<12} {score:.1%}  {bar}')

# SHAP summary plot
plt.figure(figsize=(10, 5))
shap.summary_plot(
    shap_values, scaled_data, feature_names=FEAT_LABELS,
    class_names=[segment_map.get(i, f'C{i}') for i in range(optimal_k)],
    plot_type='bar', show=False
)
plt.title('Mean |SHAP| Value per RFM Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance + centroid profile
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sorted_idx = np.argsort(combined_imp)[::-1]
sorted_feats = [FEAT_LABELS[i] for i in sorted_idx]
sorted_imps  = combined_imp[sorted_idx]
axes[0].barh(sorted_feats[::-1], sorted_imps[::-1],
             color=plt.cm.viridis(np.linspace(0.3, 0.9, len(sorted_feats)))[::-1],
             edgecolor='white')
axes[0].set_title('Feature Importance (Normalised SHAP)', fontsize=13, fontweight='bold')
for i, (feat, val) in enumerate(zip(sorted_feats[::-1], sorted_imps[::-1])):
    axes[0].text(val+0.005, i, f'{val:.1%}', va='center', fontsize=10)

# Centroid profile — using rfm_df medians per segment
centroid_data = rfm_df.groupby('Segment')[['Recency','Frequency','Monetary']].median()
# Normalise for visual comparison
centroid_norm = (centroid_data - centroid_data.min()) / (centroid_data.max() - centroid_data.min() + 1e-10)
x = np.arange(len(FEAT_LABELS))
w = 0.8 / len(segments)
for i, seg in enumerate(segments):
    if seg not in centroid_norm.index: continue
    axes[1].bar(x + i*w, centroid_norm.loc[seg, FEAT_LABELS], w,
                label=seg[:18], color=seg_color_map.get(seg, PALETTE[i]), edgecolor='white')
axes[1].set_title('Segment RFM Centroid Profiles (Normalised)', fontsize=12, fontweight='bold')
axes[1].set_xticks(x + w*(len(segments)-1)/2)
axes[1].set_xticklabels(FEAT_LABELS, fontsize=10)
axes[1].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ SHAP complete')

  EXPLAINABLE AI (SHAP) — on customer RFM vectors

  Surrogate RF accuracy: 100.0%

  Feature Importance (mean |SHAP|):
    Recency      50.1%  ████████████████████
    Monetary     49.9%  ███████████████████
    Frequency    0.0%  

✅ SHAP complete


## Cell 19: Revenue Pareto & Business Insights

In [ ]:
print('=' * 60)
print('  BUSINESS INSIGHTS & REVENUE PARETO')
print('=' * 60)

sorted_segs = sorted(segment_insights.items(), key=lambda x: x[1]['total_revenue'], reverse=True)
total_revenue = sum(info['total_revenue'] for _, info in sorted_segs)

for seg, info in sorted_segs:
    rev_share = info['total_revenue'] / total_revenue * 100
    print(f"\n  {'='*50}")
    print(f"  {seg.upper()}")
    print(f"  {'='*50}")
    print(f"  Customers: {info['customer_count']} ({info['customer_pct']}%)")
    print(f"  Avg R/F/M: {info['avg_recency']}d / {info['avg_frequency']} orders / {CUR} {info['avg_monetary']:,.2f}")
    print(f"  Revenue:   {CUR} {info['total_revenue']:,.2f} ({rev_share:.1f}% of total)")

# Revenue Pareto
seg_names = [s[0] for s in sorted_segs]
seg_revs  = [s[1]['total_revenue'] for s in sorted_segs]
cum_pcts  = [sum(seg_revs[:i+1])/total_revenue*100 for i in range(len(seg_revs))]
pareto_colors = [seg_color_map.get(s, '#3498DB') for s in seg_names]

fig, ax1 = plt.subplots(figsize=(12, 6))
bars = ax1.bar(seg_names, seg_revs, color=pareto_colors, edgecolor='white', linewidth=2)
ax1.set_ylabel(f'Revenue ({CUR})', fontsize=12); ax1.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, seg_revs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+total_revenue*0.002,
             f'{CUR} {val:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(seg_names, cum_pcts, 'o-', color='#E74C3C', linewidth=2.5, markersize=8)
ax2.axhline(y=80, color='grey', linestyle='--', alpha=0.5, label='80% line')
ax2.set_ylabel('Cumulative %', fontsize=12, color='#E74C3C')
for xv, yv in zip(seg_names, cum_pcts):
    ax2.annotate(f'{yv:.0f}%', (xv, yv), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=10, fontweight='bold', color='#E74C3C')
ax2.legend(fontsize=10)
plt.title('Revenue Pareto Analysis (80/20 Rule)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/revenue_pareto.png', dpi=150, bbox_inches='tight')
plt.show()

# Priority matrix (customers vs revenue)
fig, ax = plt.subplots(figsize=(10, 8))
for seg, info in segment_insights.items():
    color = seg_color_map.get(seg, '#3498DB')
    size = max(info['customer_count'] * 5, 100)
    ax.scatter(info['customer_count'], info['total_revenue'], s=size, c=color,
               alpha=0.7, edgecolors='black', linewidth=1.5)
    ax.annotate(seg, (info['customer_count'], info['total_revenue']),
                textcoords='offset points', xytext=(8, 4), fontsize=10, fontweight='bold')
ax.set_xlabel('Number of Customers', fontsize=12)
ax.set_ylabel(f'Total Revenue ({CUR})', fontsize=12)
ax.set_title('Strategic Priority Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/charts/priority_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Business insights complete')

  BUSINESS INSIGHTS & REVENUE PARETO

  HIBERNATING
  Customers: 274 (34.8%)
  Avg R/F/M: 293.6d / 1.0 orders / GHS 560.38
  Revenue:   GHS 153,543.82 (48.5% of total)

  POTENTIAL LOYALISTS (1)
  Customers: 132 (16.8%)
  Avg R/F/M: 187.4d / 1.0 orders / GHS 517.70
  Revenue:   GHS 68,336.00 (21.6% of total)

  POTENTIAL LOYALISTS (2)
  Customers: 42 (5.3%)
  Avg R/F/M: 84.0d / 1.0 orders / GHS 1,154.17
  Revenue:   GHS 48,475.00 (15.3% of total)

  ABOUT TO SLEEP (1)
  Customers: 210 (26.6%)
  Avg R/F/M: 278.3d / 1.0 orders / GHS 139.95
  Revenue:   GHS 29,388.73 (9.3% of total)

  ABOUT TO SLEEP (2)
  Customers: 130 (16.5%)
  Avg R/F/M: 184.6d / 1.0 orders / GHS 130.15
  Revenue:   GHS 16,920.00 (5.3% of total)

✅ Business insights complete


## Cell 20: Generate PDF Business Report

In [ ]:
print('=' * 60)
print('  GENERATING BUSINESS PDF REPORT')
print('=' * 60)

def _c(t):
    """Safe latin-1 encode for fpdf2."""
    return str(t).replace('\u2014','-').replace('\u2013','-').replace('\u2018',"'").replace('\u2019',"'").replace('\u201c','"').replace('\u201d','"').replace('\u00b1','+/-').encode('latin-1','replace').decode('latin-1')

class UniversalReport(FPDF):
    def header(self):
        self.set_font('Helvetica','B',10); self.set_text_color(100,100,100)
        self.cell(0,8,_c('Customer360 - RFM Segmentation Report'),0,0,'L')
        self.cell(0,8,datetime.now().strftime('%B %d, %Y'),0,1,'R')
        self.set_draw_color(41,128,185); self.line(10,18,200,18); self.ln(5)
    def footer(self):
        self.set_y(-15); self.set_font('Helvetica','I',8); self.set_text_color(128,128,128)
        self.cell(0,10,f'Page {self.page_no()}/{{nb}} | Customer360 | Confidential',0,0,'C')
    def ch(self,t):
        self.set_font('Helvetica','B',16); self.set_text_color(41,128,185)
        self.cell(0,12,_c(t),0,1)
        self.set_draw_color(41,128,185); self.line(10,self.get_y(),200,self.get_y()); self.ln(6)
    def sub(self,t):
        self.set_font('Helvetica','B',11); self.set_text_color(41,128,185)
        self.cell(0,8,_c(t),0,1); self.ln(1)
    def body(self,t):
        self.set_font('Helvetica','',10); self.set_text_color(60,60,60)
        self.multi_cell(0,5,_c(t)); self.ln(2)
    def bullet(self,t):
        self.set_font('Helvetica','',10); self.set_text_color(60,60,60)
        self.cell(8,5,''); self.multi_cell(0,5,_c(f'- {t}')); self.ln(1)
    def kpi(self,items):
        bw=180//len(items)
        for lbl,val in items:
            self.set_fill_color(41,128,185); self.set_text_color(255,255,255)
            self.set_font('Helvetica','B',13)
            self.cell(bw,10,_c(str(val)),1,0,'C',True)
        self.ln()
        for lbl,val in items:
            self.set_fill_color(230,240,255); self.set_text_color(52,73,94)
            self.set_font('Helvetica','',8)
            self.cell(bw,6,_c(lbl),1,0,'C',True)
        self.ln(7)
    def img(self,path,w=170):
        if os.path.exists(path):
            if 297-self.get_y()-25 < 70: self.add_page()
            try: self.image(path,x=15,w=w); self.ln(6)
            except: pass

pdf = UniversalReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=25)

# Cover
pdf.add_page(); pdf.ln(20)
pdf.set_font('Helvetica','B',34); pdf.set_text_color(41,128,185)
pdf.cell(0,14,'Customer360',0,1,'C')
pdf.set_font('Helvetica','B',17); pdf.set_text_color(52,73,94)
pdf.cell(0,9,'RFM Customer Segmentation Report',0,1,'C')
pdf.set_font('Helvetica','',12); pdf.set_text_color(100,100,100)
pdf.cell(0,7,f'Business: {BIZ}  |  Currency: {CUR}',0,1,'C')
pdf.cell(0,7,f'{len(rfm_df):,} unique customers  |  {len(df):,} transactions',0,1,'C')
pdf.cell(0,7,f'{optimal_k} segments identified via {best_algo_name}',0,1,'C')
pdf.cell(0,7,f'Date: {datetime.now().strftime("%B %d, %Y")}',0,1,'C')

# Executive Summary
pdf.add_page()
pdf.ch('Executive Summary')
pdf.kpi([
    ('Unique Customers', f'{len(rfm_df):,}'),
    ('Segments Found', str(optimal_k)),
    ('Total Revenue', f'{CUR} {total_revenue:,.0f}'),
    ('Best Algorithm', best_algo_name),
    ('Silhouette Score', f'{final_sil:.3f}'),
])

# Segment summary table
pdf.sub('Segment Overview')
pdf.set_font('Helvetica','B',9)
cw = [45,20,28,28,20,20,22]
for h,w in zip(['Segment','Customers','Avg Recency','Avg Monetary','Avg F','Disc%','Rev Share'],cw):
    pdf.set_fill_color(41,128,185); pdf.set_text_color(255,255,255)
    pdf.cell(w,7,h,1,0,'C',True)
pdf.ln()
for i,(seg,info) in enumerate(sorted_segs):
    rev_pct = info['total_revenue']/total_revenue*100
    rc = [(240,248,255),(255,255,255)][i%2]
    pdf.set_fill_color(*rc); pdf.set_text_color(40,40,40); pdf.set_font('Helvetica','',9)
    pdf.cell(cw[0],6,seg[:22],1,0,'L',True)
    pdf.cell(cw[1],6,str(info['customer_count']),1,0,'C',True)
    pdf.cell(cw[2],6,f'{info["avg_recency"]}d',1,0,'C',True)
    pdf.cell(cw[3],6,f'{CUR} {info["avg_monetary"]:,.0f}',1,0,'C',True)
    pdf.cell(cw[4],6,str(info['avg_frequency']),1,0,'C',True)
    pdf.cell(cw[5],6,f'{info["discount_rate"]}%',1,0,'C',True)
    pdf.cell(cw[6],6,f'{rev_pct:.1f}%',1,0,'C',True)
    pdf.ln()
pdf.ln(4)
pdf.img(f'{OUTPUT_DIR}/charts/segment_distribution.png')

# Per-segment pages
pal = [(39,174,96),(41,128,185),(142,68,173),(230,126,34),(192,57,43),(22,160,133),(52,73,94),(243,156,18)]
for i,(seg,info) in enumerate(sorted_segs):
    pdf.add_page()
    color = pal[i%len(pal)]
    rev_share = info['total_revenue']/total_revenue*100
    pdf.set_font('Helvetica','B',18); pdf.set_text_color(*color)
    pdf.cell(0,10,f'[{i+1}]  {seg}',0,1)
    pdf.set_draw_color(*color); pdf.line(10,pdf.get_y(),200,pdf.get_y()); pdf.ln(4)
    pdf.kpi([
        ('Customers',    str(info['customer_count'])),
        ('Avg Recency',  f'{info["avg_recency"]} days'),
        ('Avg Frequency',f'{info["avg_frequency"]} orders'),
        ('Avg Monetary',  f'{CUR} {info["avg_monetary"]:,.0f}'),
        ('Revenue Share', f'{rev_share:.1f}%'),
    ])
    pdf.sub('RFM Profile')
    pdf.bullet(f'Recency — buys on average {info["avg_recency"]} days ago')
    pdf.bullet(f'Frequency — {info["avg_frequency"]} orders on average per customer')
    pdf.bullet(f'Monetary — average {CUR} {info["avg_monetary"]:,.2f} lifetime spend per customer')
    if info['discount_rate'] > 10:
        pdf.bullet(f'Price-sensitive: uses discounts in {info["discount_rate"]}% of transactions')
    else:
        pdf.bullet(f'Brand-loyal: low discount usage ({info["discount_rate"]}%)')
    if info['top_category'] != 'N/A':
        pdf.bullet(f'Top category: {info["top_category"]}')
    if info['top_payment'] != 'N/A':
        pdf.bullet(f'Preferred payment: {info["top_payment"]}')

# Technical appendix
pdf.add_page()
pdf.ch('Technical Appendix')
pdf.sub('RFM Methodology')
pdf.body('Recency = days since last transaction (lower = better). Frequency = unique order count per customer. Monetary = total lifetime spend per customer.')
pdf.sub('Algorithm Comparison')
for algo,res in algo_comparison.items():
    mark = '  <- SELECTED' if algo == best_algo_name else ''
    pdf.body(f'{algo}: Sil={res["Silhouette"]:.4f} | DB={res["Davies-Bouldin"]:.4f} | CH={res["Calinski-H"]:.1f}{mark}')
pdf.sub('Statistical Validation')
for feat,res in anova_results.items():
    pdf.body(f'ANOVA {feat}: F={res["F"]:.2f}, p={res["p"]:.2e} ({"Significant" if res["sig"] else "Not significant"})')
pdf.body(f'ARI Stability: {avg_ari:.4f} +/- {std_ari:.4f} ({stability})')
pdf.sub('SHAP Feature Importance')
for feat,imp in sorted(zip(FEAT_LABELS, combined_imp), key=lambda x: x[1], reverse=True):
    pdf.body(f'{feat}: {imp:.1%}')
pdf.img(f'{OUTPUT_DIR}/charts/rfm_violin_plots.png')
pdf.img(f'{OUTPUT_DIR}/charts/radar_chart.png')
pdf.img(f'{OUTPUT_DIR}/charts/shap_bar.png')
pdf.img(f'{OUTPUT_DIR}/charts/algorithm_comparison.png')
pdf.img(f'{OUTPUT_DIR}/charts/statistical_validation.png')

report_path = f'{OUTPUT_DIR}/Customer360_RFM_Report.pdf'
pdf.output(report_path)
print(f'\n✅ PDF saved: {report_path}')

  GENERATING BUSINESS PDF REPORT

✅ PDF saved: /content/customer360_results/Customer360_RFM_Report.pdf


## Cell 21: Save Results & Download

In [ ]:
print('=' * 60)
print('  SAVING & DOWNLOADING RESULTS')
print('=' * 60)

# Save customer-level RFM segmented CSV
rfm_output = rfm_df[['CustomerID','Recency','Frequency','Monetary','Cluster','Segment']].copy()
rfm_csv = f'{OUTPUT_DIR}/customer_segments_rfm.csv'
rfm_output.to_csv(rfm_csv, index=False)
print(f'\n  Customer RFM CSV saved: {rfm_csv}  ({len(rfm_output):,} customers)')

# Save transaction-level CSV with segment labels joined
save_cols = ['Cluster', 'Segment']
for col_key in ['order','customer','date','product','category','qty','price','revenue','discount','payment']:
    col = SCHEMA.get(col_key)
    if col and col in df.columns:
        save_cols.insert(0, col)
save_cols = list(dict.fromkeys(save_cols))
txn_csv = f'{OUTPUT_DIR}/segmented_transactions.csv'
df[save_cols].to_csv(txn_csv, index=False)
print(f'  Transaction CSV saved:  {txn_csv}  ({len(df):,} rows)')

# Final summary
print(f'\n{"="*60}')
print('  PIPELINE COMPLETE')
print(f'{"="*60}')
print(f'  Dataset         : {BIZ}')
print(f'  Unique customers: {len(rfm_df):,}')
print(f'  Transactions    : {len(df):,}')
print(f'  Segments        : {optimal_k}  ({best_algo_name})')
print(f'  Segment names   : {segments}')
print(f'  Silhouette      : {final_sil:.4f}')
print(f'  Davies-Bouldin  : {final_db:.4f}')
print(f'  Calinski-H      : {final_ch:.1f}')
print(f'  ARI Stability   : {stability} ({avg_ari:.4f})')
print(f'  Total Revenue   : {CUR} {total_revenue:,.2f}')

# Trigger downloads
files.download(f'{OUTPUT_DIR}/Customer360_RFM_Report.pdf')
files.download(rfm_csv)
files.download(txn_csv)
print('\n✅ Downloads triggered!')

  SAVING & DOWNLOADING RESULTS

  Customer RFM CSV saved: /content/customer360_results/customer_segments_rfm.csv  (788 customers)
  Transaction CSV saved:  /content/customer360_results/segmented_transactions.csv  (1,504 rows)

  PIPELINE COMPLETE
  Dataset         : Fashion Retail
  Unique customers: 788
  Transactions    : 1,504
  Segments        : 5  (K-Means)
  Segment names   : ['About to Sleep (1)', 'Potential Loyalists (1)', 'Hibernating', 'Potential Loyalists (2)', 'About to Sleep (2)']
  Silhouette      : 0.4245
  Davies-Bouldin  : 0.7746
  Calinski-H      : 662.6
  ARI Stability   : Excellent (1.0000)
  Total Revenue   : GHS 316,663.55


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Downloads triggered!


In [26]:
import zipfile

bundle_path = f'{OUTPUT_DIR}/customer360_model_bundle.zip'
bundle_files = [
    f'{OUTPUT_DIR}/scaler.pkl',
    f'{OUTPUT_DIR}/segment_map.json',
    f'{OUTPUT_DIR}/cluster_rfm_profile.csv',
]
with zipfile.ZipFile(bundle_path, 'w') as zf:
    for fp in bundle_files:
        if os.path.exists(fp):
            zf.write(fp, os.path.basename(fp))

files.download(bundle_path)
print('✅ Model bundle downloaded: customer360_model_bundle.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model bundle downloaded: customer360_model_bundle.zip
